In [ ]:
import sys
from pathlib import Path

# Allow importing project config
sys.path.insert(0, str(Path("../..").resolve()))
from configs.config import JSON_CHUNKS_DIR, FAISS_INDEX_DIR

# --- Papermill parameters (overwritten at runtime) ---
input_path       = str(JSON_CHUNKS_DIR / "rag_chunks_split_langchain.jsonl")
faiss_output_dir = str(FAISS_INDEX_DIR)
embedding_model  = "all-MiniLM-L6-v2"
experiment_name  = "default_run"
run_id           = "default_run_id"

In [ ]:
# Incluir esto al comienzo del notebook (después de la celda de parámetros si usas papermill)
import mlflow

# Asegurar que estamos en el run correcto sin iniciar uno nuevo
if mlflow.active_run() is None and "run_id" in globals():
    mlflow.start_run(run_id=run_id)


In [ ]:
import os
import json
import mlflow
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from huggingface_hub import login

from src.utils import build_embedder

hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN", "")
if hf_token:
    login(hf_token)

mlflow.log_params({
    "input_path":      input_path,
    "faiss_output_dir": faiss_output_dir,
    "embedding_model": embedding_model,
})

os.makedirs(faiss_output_dir, exist_ok=True)

documents = []
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        record  = json.loads(line)
        content = record.pop("content")
        documents.append(Document(page_content=content, metadata=record))

total_docs = len(documents)
print(f"Total documents loaded: {total_docs}")
mlflow.log_metric("total_documents", total_docs)

# build_embedder returns an embedder whose embed_query adds
# the BGE prefix automatically; documents are indexed as-is.
embedder = build_embedder(embedding_model)

vectorstore = FAISS.from_documents(documents, embedder)
vectorstore.save_local(faiss_output_dir)
print(f"FAISS index saved to: {faiss_output_dir}")

mlflow.log_artifacts(faiss_output_dir, artifact_path="faiss_index")
mlflow.end_run()